In [3]:
from Tokenizer.BpeToken import BpeTokenizer
from config.ConfigFile import Config
from datasets import load_dataset
import argparse
import time
import torch
import numpy as np
import os
from tqdm import tqdm

def load_dataset_all(download_data_key,datasets="wikitext"):

    if  download_data_key[datasets][1] is not None :
        dataset = load_dataset(download_data_key[datasets][0], download_data_key[datasets][1])
    else:
        dataset = load_dataset(download_data_key[datasets][0])

    split_name = "train" if "train" in dataset else list(dataset.keys())[0]
    #train_texts = [line for line in dataset[split_name]["text"] if line.strip() != ""]
    texts = np.array(dataset[split_name]["text"])
    mask = np.char.strip(texts) != ""
    train_texts = texts[mask].tolist()
    #train_texts = dataset[split_name]["text"]
    return train_texts

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

## Config files
config_parser=Config()
download_data_key = config_parser.download_data_key
dataset = config_parser.dataset
path = config_parser.path
data_path= config_parser.data_path
os.makedirs(data_path ,exist_ok=True)
## Load data sets
train_texts = load_dataset_all(download_data_key,dataset)

print(train_texts[:5])

## config BpeTokenizer
bpetoken = BpeTokenizer(train_texts)

## Save tokenizer
bpetoken.save(path)

## Load tokenizer 
bpe2 = BpeTokenizer.load(path)

# Test encode-decode
sample = "Hello, world!"
ids = bpe2.encode(sample)
decoded = bpe2.decode(ids)

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

In [3]:
print("Original:", sample)
print("Token IDs:", ids)
print("Decoded:", decoded)

Original: Hello, world!
Token IDs: [2, 43, 419, 82, 15, 1279, 4, 3]
Decoded: <BOS>Hello, world!<EOS>


In [14]:
print(f"full text of all concatenated")
separator = '\n\n'  # Or ' ' if lines are sentences; preserves article structure
all_data = separator.join([t.strip() for t in train_texts if t.strip()])
print(f"few rows train_texts :  {train_texts[:3]}")
print(f"few rows all_data : {all_data[:300]}")

print("Encoding full text...")
start_time = time.time()
full_encoded = bpe2.tokenizer.encode(all_data)  # Single encode; adds no BOS/EOS here (handle in chunks)
torch.save(full_encoded, data_path + "full_encoded.pt")
full_ids = full_encoded.ids
torch.save(full_ids, data_path + "full_ids.pt")
elapsed = time.time() - start_time
print(f"Full encode time: {elapsed:.2f} sec")

full text of all concatenated
few rows train_texts :  [' = Valkyria Chronicles III = \n', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple ad

In [15]:
encoded_dataset = []
max_length = 512  # From your training config; adjust as needed
stride = int(max_length * 0.9)  # 10% overlap; tune 0.8-0.95 for more/less density
for i in tqdm(range(0, len(full_ids) - max_length + 1, stride), desc="Chunking sequences"):
    chunk_ids = full_ids[i:i + max_length]
    if len(chunk_ids) < max_length * 0.5:  # Skip tiny tail
        break
    # Wrap with specials (mimic per-text)
    bos_id = bpe2.tokenizer.token_to_id("<BOS>")
    eos_id = bpe2.tokenizer.token_to_id("<EOS>")
    wrapped_chunk = [bos_id] + chunk_ids + [eos_id]
    encoded_dataset.append(wrapped_chunk[:max_length])  # Trim if over (rare)

print(f"Created {len(encoded_dataset)} sequences (avg len: {np.mean([len(seq) for seq in encoded_dataset]):.0f})")

# Save (use fast if available, but single encode is already fast)
start_time = time.time()
torch.save(encoded_dataset, data_path + "encoded_dataset_long.pt")
elapsed = time.time() - start_time
print(f"Save time: {elapsed:.2f} sec")
print(f"Few rows: {encoded_dataset[:3]}")


Chunking sequences: 100%|██████████| 5964/5964 [00:00<00:00, 57012.17it/s]


Created 5964 sequences (avg len: 512)
Save time: 0.46 sec
Few rows: [[2, 2, 32, 7787, 3662, 6286, 596, 2990, 241, 161, 161, 54, 216, 77, 4765, 829, 7787, 3662, 431, 469, 612, 6884, 415, 207, 6286, 596, 314, 2749, 469, 162, 154, 171, 102, 153, 195, 115, 4130, 109, 3986, 115, 4449, 97, 3986, 107, 4449, 196, 3986, 101, 3986, 106, 4449, 98, 22, 204, 6377, 212, 7787, 3662, 221, 201, 3458, 1975, 431, 313, 204, 5102, 3245, 230, 284, 7787, 3662, 6286, 596, 2990, 2556, 1762, 204, 304, 199, 5216, 455, 1499, 280, 2333, 1467, 677, 1901, 310, 745, 3544, 228, 7517, 17, 57, 722, 277, 201, 4780, 1592, 617, 212, 7424, 623, 224, 1344, 1636, 224, 1762, 204, 325, 304, 201, 1371, 677, 224, 201, 7787, 3662, 846, 212, 2471, 1798, 226, 201, 1052, 217, 4247, 221, 5216, 455, 228, 1566, 280, 585, 6496, 284, 460, 6621, 624, 204, 201, 1425, 3376, 7455, 230, 201, 458, 677, 228, 4690, 201, 242, 321, 254, 6766, 242, 204, 199, 2734, 225, 1683, 3776, 5594, 201, 4199, 221, 4682, 430, 664, 201, 3521, 1291, 223, 1054, 519

In [17]:
len(encoded_dataset)

5964

In [19]:
encoded_dataset[:1]

[[2,
  2,
  32,
  7787,
  3662,
  6286,
  596,
  2990,
  241,
  161,
  161,
  54,
  216,
  77,
  4765,
  829,
  7787,
  3662,
  431,
  469,
  612,
  6884,
  415,
  207,
  6286,
  596,
  314,
  2749,
  469,
  162,
  154,
  171,
  102,
  153,
  195,
  115,
  4130,
  109,
  3986,
  115,
  4449,
  97,
  3986,
  107,
  4449,
  196,
  3986,
  101,
  3986,
  106,
  4449,
  98,
  22,
  204,
  6377,
  212,
  7787,
  3662,
  221,
  201,
  3458,
  1975,
  431,
  313,
  204,
  5102,
  3245,
  230,
  284,
  7787,
  3662,
  6286,
  596,
  2990,
  2556,
  1762,
  204,
  304,
  199,
  5216,
  455,
  1499,
  280,
  2333,
  1467,
  677,
  1901,
  310,
  745,
  3544,
  228,
  7517,
  17,
  57,
  722,
  277,
  201,
  4780,
  1592,
  617,
  212,
  7424,
  623,
  224,
  1344,
  1636,
  224,
  1762,
  204,
  325,
  304,
  201,
  1371,
  677,
  224,
  201,
  7787,
  3662,
  846,
  212,
  2471,
  1798,
  226,
  201,
  1052,
  217,
  4247,
  221,
  5216,
  455,
  228,
  1566,
  280,
  585,
  6496,
  284,
  460,

In [ ]:
import os
import time
import torch
import torch.nn as nn
from tqdm import tqdm

from Tokenizer.BpeToken import BpeTokenizer
from config.ConfigFile import Config
from transformer.transformer import *

# -------------------------------
# Config and setup
# -------------------------------
config = Config()
print(dict(config.__dict__.items()))

sample_method = "sample"   # "argmax" or "sample"
temperature = 0.9        # <1.0 makes output more deterministic
top_k = 50                 # keep only top K tokens
top_p = 0.6                # nucleus sampling
max_new_tokens = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------
# Load model and tokenizer
# -------------------------------
model = torch.load(os.path.join(config.data_path, "best_model_full.pt"), weights_only=False)
model.to(device)
model.eval()

tokenizer = BpeTokenizer.load(config.path)
eos_token_id = tokenizer.tokenizer.token_to_id("<EOS>")

print("Model and tokenizer loaded.")
time.sleep(1)

# -------------------------------
# Sampling helpers
# -------------------------------
def apply_temperature_and_filtering(logits, temperature=1.0, top_k=0, top_p=1.0):
    # scale by temperature
    logits = logits / max(temperature, 1e-8)

    # top-k
    if top_k > 0:
        values, _ = torch.topk(logits, top_k)
        min_values = values[:, -1].unsqueeze(-1)
        logits[logits < min_values] = -float("Inf")

    # top-p (nucleus)
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(nn.functional.softmax(sorted_logits, dim=-1), dim=-1)

        # remove tokens with cumulative prob above top_p
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[:, indices_to_remove] = -float("Inf")

    return logits

# -------------------------------
# Interactive loop
# -------------------------------
while True:
    print("\nEnter Text (or 'exit' to quit):")
    input_text = input()
    if input_text.lower() == "exit":
        break

    token_data = tokenizer.encode(input_text)
    print(f"Input '{input_text}' → tokens {token_data}")

    if eos_token_id is not None and token_data and token_data[-1] == eos_token_id:
        token_data = token_data[:-1]

    # start generating
    print("Start inference...")
    with torch.no_grad():
        for step in tqdm(range(max_new_tokens), desc="Generating Tokens", total=max_new_tokens):
            token_tensor_list_nopad = convert_listbatch_to_listtensor(
                [token_data], device, max_length=model.max_length, dtype=torch.long
            )
            if len(token_data) < model.max_length:
                token_tensor_list = pad_sequence_list(token_tensor_list_nopad, model.max_length, pad_token_id=0)
            else:
                token_tensor_list = token_tensor_list_nopad[:, -model.max_length:]

            output, _ = model(token_tensor_list)
            logits = output[:, -1, :]  # last token logits

            if sample_method == "argmax":
                next_token = torch.argmax(logits, dim=-1)
            else:
                # apply temperature + top-k/top-p filtering
                logits = apply_temperature_and_filtering(logits, temperature, top_k, top_p)
                probs = nn.functional.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).squeeze(1)

            next_token_id = next_token.item()
            token_data.append(next_token_id)

            # incremental decode
            partial_text = tokenizer.decode(token_data, skip_special_tokens=True).rstrip()
            print(f"Step {step+1}: {partial_text}")

            if eos_token_id is not None and next_token_id == eos_token_id:
                print("EOS token generated. Stopping early.")
                break

    # Final decode
    output = tokenizer.decode(token_data, skip_special_tokens=True).rstrip()
    print(f"\nFinal Output: {output}")


{'data_path': './data/', 'path': './data/tokenizer.json', 'dataset': 'wikitext', 'batch_size_token': 1024, 'download_data_key': {'wikitext': ('wikitext', 'wikitext-2-raw-v1'), 'wikitext1': ('wikitext', 'wikitext-103-raw-v1'), 'shakespeare': ('tiny_shakespeare', None), 'ptb': ('ptb_text_only', None), 'news': ('ag_news', None), 'imdb': ('imdb', None)}}
Model and tokenizer loaded.

Enter Text (or 'exit' to quit):


In [11]:

def get_sequence_size(list_seq):
    return max(len(seq) for seq in list_seq)

def convert_listbatch_to_listtensor(batch_seq,device, max_length, dtype=torch.long):
    return [torch.tensor(seq[:max_length], device=device , dtype=dtype) for seq in batch_seq]

def pad_sequence_list(vector_list,max_length,pad_token_id=0):
    return torch.stack([f.pad(vector,(0,max_length-len(vector))) for vector in vector_list],dim=0)

def get_device():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    return device
device= get_device()

Using device: cuda


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as f
data= torch.load( data_path+"encoded_dataset_long.pt")

In [17]:
k=0
batch_size=16
batch = data[k:k+batch_size]


In [18]:
max_length=512
vector_list = convert_listbatch_to_listtensor(batch, device, max_length, dtype=torch.long)

In [19]:
batch_tensor = pad_sequence_list(vector_list,max_length,pad_token_id=0)

In [26]:
batch_tensor

tensor([[    2,     2,    44,  ...,   785,  4699,  6816],
        [    2,   616,   256,  ...,   250, 13755,   397],
        [    2,   234,  2137,  ...,    44,   428,   978],
        ...,
        [    2,  9584,    17,  ...,   587,   281,   628],
        [    2,   833,   268,  ...,  2089,  5181,   548],
        [    2,   425,  1044,  ...,  5581,   197,   165]], device='cuda:0')

In [30]:
batch_tensor[:, 1:-1]

tensor([[    2,    44,  4182,  ...,   608,   785,  4699],
        [  616,   256,   204,  ...,   251,   250, 13755],
        [  234,  2137,    15,  ...,   206,    44,   428],
        ...,
        [ 9584,    17,    44,  ...,   278,   587,   281],
        [  833,   268,   868,  ...,   785,  2089,  5181],
        [  425,  1044,    39,  ...,   238,  5581,   197]], device='cuda:0')

In [112]:
import os
import time
import torch
import torch.nn as nn
from tqdm import tqdm

from Tokenizer.BpeToken import BpeTokenizer
from config.ConfigFile import Config
from transformer.transformer import *

# -------------------------------
# Config and setup
# -------------------------------
config = Config()
print(dict(config.__dict__.items()))

sample_method = "sample"   # "argmax" or "sample"
temperature = 1.0        # <1.0 makes output more deterministic
top_k = 50                 # keep only top K tokens
top_p = 0.6                # nucleus sampling
max_new_tokens = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------------------
# Load model and tokenizer
# -------------------------------
model = torch.load(os.path.join(config.data_path, "best_model_full.pt"), weights_only=False)
model.to(device)
model.eval()

tokenizer = BpeTokenizer.load(config.path)
eos_token_id = tokenizer.tokenizer.token_to_id("<EOS>")

print("Model and tokenizer loaded.")
time.sleep(1)

# -------------------------------
# Sampling helpers
# -------------------------------
def apply_temperature_and_filtering(logits, temperature=1.0, top_k=0, top_p=1.0):
    # scale by temperature
    logits = logits / max(temperature, 1e-8)

    # top-k
    if top_k > 0:
        values, _ = torch.topk(logits, top_k)
        min_values = values[:, -1].unsqueeze(-1)
        logits[logits < min_values] = -float("Inf")

    # top-p (nucleus)
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(nn.functional.softmax(sorted_logits, dim=-1), dim=-1)

        # remove tokens with cumulative prob above top_p
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[:, indices_to_remove] = -float("Inf")

    return logits

{'data_path': './data/', 'path': './data/tokenizer.json', 'dataset': 'imdb', 'batch_size_token': 1024, 'max_length': 512, 'stride': 460, 'download_data_key': {'wikitext': ('wikitext', 'wikitext-2-raw-v1'), 'wikitext1': ('wikitext', 'wikitext-103-raw-v1'), 'shakespeare': ('tiny_shakespeare', None), 'ptb': ('ptb_text_only', None), 'news': ('ag_news', None), 'imdb': ('imdb', None)}}
Model and tokenizer loaded.


In [138]:
input_text="timed and occasion"
token_data = tokenizer.encode(input_text)
print(f"Input '{input_text}' → tokens {token_data}")

Input 'timed and occasion' → tokens [2, 78, 179, 97, 136, 15897, 3]


In [114]:
token_data =[2,     2,  8872,   250,    15,  5815,   862,   479,  2873,  4536,
            96,  2659,   175,   319,    11,   680,   161,  7201,    14,    77,]

In [139]:
if eos_token_id is not None and token_data and token_data[-1] == eos_token_id:
    token_data = token_data[:-1]

In [140]:
token_data

[2, 78, 179, 97, 136, 15897]

In [141]:
model.max_length

512

In [142]:
token_tensor_list_nopad = convert_listbatch_to_listtensor(
    [token_data], device, max_length=model.max_length, dtype=torch.long
)



In [143]:
token_tensor_list_nopad

[tensor([    2,    78,   179,    97,   136, 15897], device='cuda:0')]

In [144]:
if len(token_data) < model.max_length:
    token_tensor_list = pad_sequence_list(token_tensor_list_nopad, model.max_length, pad_token_id=0)
else:
    token_tensor_list = token_tensor_list_nopad[:, -model.max_length:]

output, _ = model(token_tensor_list)
logits = output[:, -1, :]  # last token logits

In [145]:
logits

tensor([[181.5788, -59.4134,  -7.6128,  ...,   9.8463, -28.9838,  22.8442]],
       device='cuda:0', grad_fn=<SliceBackward0>)

In [146]:

next_token = torch.argmax(logits, dim=-1)

In [147]:
next_token

tensor([0], device='cuda:0')

In [148]:
tokenizer.decode([next_token])

'<PAD>'

In [154]:
# generate.py
from transformer.transformer import *
import os
import math
import torch
import torch.nn.functional as F
from typing import Optional, List, Tuple

from Tokenizer.BpeToken import BpeTokenizer    # as in your project

# -------------------------
# Utilities for top-k / top-p filtering
# -------------------------
def top_k_top_p_filtering(logits: torch.Tensor, top_k: int = 0, top_p: float = 0.0, filter_value: float = -float("Inf")) -> torch.Tensor:
    """
    logits: (vocab,)
    Returns filtered logits (same shape) where filtered tokens are set to filter_value.
    """
    top_k = int(top_k)
    if top_k > 0:
        # Remove all tokens with a probability less than the top-k tokens
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value

    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative probability above top_p
        sorted_indices_to_remove = cumulative_probs > top_p
        # shift the indices to the right to keep at least one token
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = filter_value
    return logits

# -------------------------
# Model / tokenizer loader
# -------------------------
def load_model_and_tokenizer(checkpoint_path: str,
                             full_model_path: Optional[str],
                             device: torch.device,
                             model_kwargs: dict) -> Tuple[TransformerDecoder, BpeTokenizer]:
    """
    Loads model from a state-dict checkpoint if available, else tries to load full model object.
    model_kwargs are used to instantiate TransformerDecoder when loading state_dict.
    """
    # init tokenizer (adjust constructor args if needed)
    tokenizer = BpeTokenizer.load("./data/tokenizer.json")

    # instantiate model architecture
    model = TransformerDecoder(**model_kwargs).to(device)

    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Loading state_dict from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=device)
        # if ckpt is a dict that contains 'model_state_dict'
        if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            # assume ckpt is a raw state_dict
            model.load_state_dict(ckpt)
    elif full_model_path and os.path.exists(full_model_path):
        print(f"Loading full model from {full_model_path}")
        loaded = torch.load(full_model_path, map_location=device)
        # If the saved file was the model object
        if isinstance(loaded, TransformerDecoder):
            model = loaded.to(device)
        else:
            # If someone saved a dict containing state_dict
            if isinstance(loaded, dict) and 'model_state_dict' in loaded:
                model.load_state_dict(loaded['model_state_dict'])
            else:
                # fallback: try loading as state_dict
                model.load_state_dict(loaded)
    else:
        raise FileNotFoundError("No checkpoint or full_model file found at provided paths.")

    model.eval()
    return model, tokenizer

# -------------------------
# Autoregressive generation
# -------------------------
@torch.no_grad()
def generate(
    model: TransformerDecoder,
    tokenizer: BpeTokenizer,
    prompt: List[int] or torch.Tensor,
    device: torch.device,
    max_new_tokens: int = 50,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 0.0,
    do_sample: bool = True,
    eos_token_id: Optional[int] = None,
    pad_token_id: int = 0,
    verbose: bool = False,
) -> List[int]:
    """
    Autoregressive generation without caching (model doesn't implement kv cache).
    prompt: list of token ids OR a tensor of shape (batch, seq_len)
    returns generated token ids (including prompt)
    """

    # Ensure prompt is a 2D tensor [batch_size, seq_len]
    if isinstance(prompt, list):
        prompt = torch.tensor([prompt], dtype=torch.long, device=device)
    elif isinstance(prompt, torch.Tensor):
        if prompt.dim() == 1:
            prompt = prompt.unsqueeze(0).to(device)
        else:
            prompt = prompt.to(device)
    batch_size = prompt.size(0)

    # Make working buffer
    cur_input = prompt.clone()
    generated = cur_input.tolist()  # list of lists [batch][tokens...]

    # loop
    for step in range(max_new_tokens):
        seq_len = cur_input.size(1)
        if seq_len > model.max_length:
            # trim leftmost tokens (simple sliding window). or raise error.
            cur_input = cur_input[:, -model.max_length:]
            seq_len = cur_input.size(1)

        # forward: model returns logits (batch, seq_len, vocab)
        logits, _ = model(cur_input)
        next_token_logits = logits[:, -1, :]  # (batch, vocab)

        # apply temperature
        if temperature != 1.0:
            next_token_logits = next_token_logits / temperature

        # filter logits by top_k, top_p
        filtered_logits = next_token_logits.clone()
        for b in range(batch_size):
            filtered_logits[b] = top_k_top_p_filtering(filtered_logits[b], top_k=top_k, top_p=top_p)

        if not do_sample:
            # Greedy
            next_tokens = torch.argmax(filtered_logits, dim=-1, keepdim=True)
        else:
            # Convert to probabilities
            probs = F.softmax(filtered_logits, dim=-1)
            # sample
            next_tokens = torch.multinomial(probs, num_samples=1)  # (batch, 1)

        # append tokens
        cur_input = torch.cat([cur_input, next_tokens], dim=1)
        for b in range(batch_size):
            generated[b].append(int(next_tokens[b].item()))

        # stop if all sequences produced eos
        if eos_token_id is not None:
            done_mask = [generated[b][-1] == eos_token_id for b in range(batch_size)]
            if all(done_mask):
                if verbose:
                    print(f"Stopping at step {step} because all sequences generated EOS.")
                break

    # return generated lists (batch)
    return generated


In [157]:
device = torch.device(device)

# model construction kwargs -- MUST match how you created the model for training
model_kwargs = {
    "vocab_size": 16384,    # override with your tokenizer vocab size or supply dynamically
    "embed_dim": 768,
    "num_heads": 4,
    "ffn_dim": 768*4,
    "num_blocks": 4,
    "max_length": 512,
    "dropout": 0.1,
    "type_pos_emb": "learnable",
    "tied_emb": 1
}

In [158]:
model, tokenizer = load_model_and_tokenizer("./data/best_model_dict.pt", "./data/best_model_full.pt", device, model_kwargs)

Per Head the embedded are : 192.0
Per Head the embedded are : 192.0
Per Head the embedded are : 192.0
Per Head the embedded are : 192.0
Loading state_dict from ./data/best_model_dict.pt


In [169]:
prompt_text=" from my video store because of all the controversy "

In [170]:
input_ids = tokenizer.encode(prompt_text)
input_ids

[2, 354, 433, 1568, 3213, 544, 191, 330, 165, 14365, 135, 3]

In [ ]:



eos_id = getattr(tokenizer, "eos_token_id", 3)
pad_id = getattr(tokenizer, "pad_token_id", 0)

generated_ids_batch = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=input_ids[:-1],
    device=device,
    max_new_tokens=5,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample="store_true",
    eos_token_id=eos_id,
    pad_token_id=pad_id,
    verbose=True
)

# decode and print (supports batch)
for i, gen_ids in enumerate(generated_ids_batch):
    # Trim padding if any and decode
    try:
        text = tokenizer.decode(gen_ids)  # most tokenizers provide decode(list_of_ids)
    except Exception:
        # fallback: join token-level decode
        text = ""
        for tok_id in gen_ids:
            try:
                text += tokenizer.id_to_token(tok_id)  # adjust to your tokenizer API
            except Exception:
                text += f"<{tok_id}>"
    print(f"\n=== Generated #{i} ===\n{text}\n")



=== Generated #0 ===
<BOS> from my video store because of all the controversy  and and is and and



In [173]:
import torch

In [175]:
dt=torch.load("./data/full_encoded.pt",weights_only=False)

In [176]:
dt

Encoding(num_tokens=7676732, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [177]:
dt.tokens

['<BOS>',
 'I',
 'Ġrented',
 'ĠI',
 'ĠAM',
 'ĠC',
 'U',
 'RI',
 'OUS',
 '-',
 'Y',
 'ELL',
 'OW',
 'Ġfrom',
 'Ġmy',
 'Ġvideo',
 'Ġstore',
 'Ġbecause',
 'Ġof',
 'Ġall',
 'Ġthe',
 'Ġcontroversy',
 'Ġthat',
 'Ġsurrounded',
 'Ġit',
 'Ġwhen',
 'Ġit',
 'Ġwas',
 'Ġfirst',
 'Ġreleased',
 'Ġin',
 'Ġ196',
 '7',
 '.',
 'ĠI',
 'Ġalso',
 'Ġheard',
 'Ġthat',
 'Ġat',
 'Ġfirst',
 'Ġit',
 'Ġwas',
 'Ġse',
 'ized',
 'Ġby',
 'ĠU',
 '.',
 'S',
 '.',
 'Ġcust',
 'oms',
 'Ġif',
 'Ġit',
 'Ġever',
 'Ġtried',
 'Ġto',
 'Ġenter',
 'Ġthis',
 'Ġcountry',
 ',',
 'Ġtherefore',
 'Ġbeing',
 'Ġa',
 'Ġfan',
 'Ġof',
 'Ġfilms',
 'Ġconsidered',
 'Ġ"',
 'cont',
 'ro',
 'vers',
 'ial',
 '"',
 'ĠI',
 'Ġreally',
 'Ġhad',
 'Ġto',
 'Ġsee',
 'Ġthis',
 'Ġfor',
 'Ġmyself',
 '.<',
 'br',
 'Ġ/><',
 'br',
 'Ġ/>',
 'The',
 'Ġplot',
 'Ġis',
 'Ġcentered',
 'Ġaround',
 'Ġa',
 'Ġyoung',
 'ĠSwedish',
 'Ġdrama',
 'Ġstudent',
 'Ġnamed',
 'ĠLena',
 'Ġwho',
 'Ġwants',
 'Ġto',
 'Ġlearn',
 'Ġeverything',
 'Ġshe',
 'Ġcan',
 'Ġabout',
 'Ġlife',
 '.',